In [1]:
2+2

4

In [27]:
import os
import json
from pathlib import Path

import random
import pandas as pd
import numpy as np
# import torch
import seaborn as sns
import matplotlib.pyplot as plt

import ast
import re
import shutil

In [3]:
# Setting the max characters to be displayed
pd.set_option('display.max_colwidth', None)

# display full dataframe pandas / output window widened
pd.set_option('display.width', 10000) # None

# Setting the precision for float columns
pd.set_option('display.precision', 2)

In [4]:
result = pd.read_parquet("logits_output_meta.parquet")
print(result.shape)
del result['top5_classes']
result.head(4)

(216, 12)


,video_id,cam,session,template,label,top1_pred,top1_class,top1_conf,top5_preds,correct_top1,correct_top5
0,260930-122254,cam0,plug_2,Putting [something] next to [something],107,107,Putting [something] next to [something],0.41,"[107, 114, 42, 19, 109]",True,True
1,260930-122254,cam1,plug_2,Putting [something] next to [something],107,107,Putting [something] next to [something],0.74,"[107, 19, 114, 42, 109]",True,True
2,260930-122254,cam2,plug_2,Putting [something] next to [something],107,105,Putting [something] in front of [something],0.75,"[105, 18, 114, 42, 125]",False,False
3,260930-122317,cam0,plug_2,Putting [something] next to [something],107,107,Putting [something] next to [something],0.41,"[107, 42, 109, 41, 112]",True,True


In [5]:
d = np.load("logits_output.npz", allow_pickle=True)
print(d["logits"].shape)   # (150, 174)
d

(216, 174)


NpzFile 'logits_output.npz' with keys: logits, probs, labels, video_ids, cams...

# main result

In [6]:
from sklearn.metrics import balanced_accuracy_score
    
n = len(result)
top1_acc  = result["correct_top1"].mean()
top5_acc  = result["correct_top5"].mean()
ba_top1   = balanced_accuracy_score(result["label"], result["top1_pred"])

# Top-5 balanced: credit correct if label is anywhere in top-5
top5_pred_adj = [
    row["label"] if row["correct_top5"] else row["top5_preds"][0]
    for _, row in result.iterrows()
]
ba_top5 = balanced_accuracy_score(result["label"], top5_pred_adj)

my_metrics = {
    "total_evaluated":        n,
    "top1_accuracy":          round(top1_acc, 6),
    "top1_balanced_accuracy": round(ba_top1,  6),
    "top5_accuracy":          round(top5_acc, 6),
    "top5_balanced_accuracy": round(ba_top5,  6),
}
my_metrics

/user/h.siddiqui/u27673/.conda/envs/vbbf/lib/python3.12/site-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/user/h.siddiqui/u27673/.conda/envs/vbbf/lib/python3.12/site-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


{'total_evaluated': 216,
 'top1_accuracy': np.float64(0.560185),
 'top1_balanced_accuracy': np.float64(0.567901),
 'top5_accuracy': np.float64(0.907407),
 'top5_balanced_accuracy': np.float64(0.919753)}

- cam0 = birds eye
- cam1 = POV
- cam2 = side view

In [7]:
result.groupby("cam")[["correct_top1", "correct_top5"]].mean().rename(
        columns={"correct_top1": "top1_acc", "correct_top5": "top5_acc"}
)

,top1_acc,top5_acc
cam,,
cam0,0.54,0.82
cam1,0.69,0.99
cam2,0.44,0.92


In [8]:
result.groupby("template")[["correct_top1", "correct_top5"]].mean().rename(
        columns={"correct_top1": "top1_acc", "correct_top5": "top5_acc"}
).sort_values("template")

,top1_acc,top5_acc
template,,
Attaching [something] to [something],0.89,1.00
Moving [something] up,0.22,0.72
Piling [something] up,0.11,1.00
Plugging [something] into [something],0.89,1.00
Pushing [something] with [something],0.75,1.00
Putting [something] next to [something],0.58,0.94
Putting [something] on a surface,0.78,0.86
Putting [something] onto [something],0.44,0.97
Showing a photo of [something] to the camera,0.44,0.78


In [10]:
result[result.cam=='cam1'].groupby("template")[["correct_top1", "correct_top5"]].mean().rename(
        columns={"correct_top1": "top1_acc", "correct_top5": "top5_acc"}
).sort_values("template")

,top1_acc,top5_acc
template,,
Attaching [something] to [something],0.4,1.0
Moving [something] up,0.2,1.0
Piling [something] up,0.0,1.0
Plugging [something] into [something],1.0,1.0
Pushing [something] with [something],1.0,1.0
Putting [something] next to [something],0.6,1.0
Putting [something] on a surface,0.6,1.0
Putting [something] onto [something],0.6,1.0
Showing a photo of [something] to the camera,0.8,0.8


In [18]:
result[result.cam=='cam1'].groupby("session").agg(
    top1_acc = ("correct_top1", "mean"),
    top5_acc = ("correct_top5", "mean")
)

,top1_acc,top5_acc
session,,
attaching_1,1.00,1.00
books_2,0.40,0.93
lady_2,0.67,1.00
magnet_2,0.87,1.00
piling_1,0.00,1.00
plug_2,0.80,1.00
plugging_1,1.00,1.00
showing_photo_of_1,1.00,1.00


# Granular analysis

##### Distractor objects -> expected spike in action/verb -- P(Action|Noun) estimate
- Plug -> Plugging [something] into [something] -- 55%
- lady/person -> Showing a photo of [something] to the camera -- 96%
- magnet -> Attaching [something] to [something] -- 60%
- Books -> Piling [something] up -- 55%
- (~) Paper -> Tearing [something] into two pieces -- 25%

##### id2label

In [19]:
with open('../../data/something-something-v2-id2label.json', 'r') as json_file:
    id2label_dict = json.load(json_file)
    label2id_dict = {v: k for k, v in id2label_dict.items()}


print(list(id2label_dict.items())[0:2])
print(list(label2id_dict.items())[0:2])


[('0', 'Approaching [something] with your camera'), ('1', 'Attaching [something] to [something]')]
[('Approaching [something] with your camera', '0'), ('Attaching [something] to [something]', '1')]


In [20]:
result.head(1)

,video_id,cam,session,template,label,top1_pred,top1_class,top1_conf,top5_preds,correct_top1,correct_top5
0,260930-122254,cam0,plug_2,Putting [something] next to [something],107,107,Putting [something] next to [something],0.41,"[107, 114, 42, 19, 109]",True,True


In [21]:
df = result[['video_id',	'cam'	,'session'	,'template',	'label'	,'top1_pred','top1_conf','correct_top1','correct_top5']]
print(df.shape)
df.head(2)

(216, 9)


,video_id,cam,session,template,label,top1_pred,top1_conf,correct_top1,correct_top5
0,260930-122254,cam0,plug_2,Putting [something] next to [something],107,107,0.41,True,True
1,260930-122254,cam1,plug_2,Putting [something] next to [something],107,107,0.74,True,True


In [22]:
# Define class probability column names
class_cols = [f'prob_{i}' for i in range(174)]

# Convert probabilities array into a DataFrame using matching index
probs_df = pd.DataFrame(d['probs'], columns=class_cols, index=df.index)

# Combine metadata with class probabilities
df = pd.concat([df, probs_df], axis=1)

pd.set_option('display.float_format',  '{:,.6f}'.format)

df.head(2)

,video_id,cam,session,template,label,top1_pred,top1_conf,correct_top1,correct_top5,prob_0,...,prob_164,prob_165,prob_166,prob_167,prob_168,prob_169,prob_170,prob_171,prob_172,prob_173
0,260930-122254,cam0,plug_2,Putting [something] next to [something],107,107,0.407918,True,True,0.000083,...,0.000481,0.000127,0.000592,0.000040,0.000161,0.000007,0.000072,0.000350,0.000003,0.000016
1,260930-122254,cam1,plug_2,Putting [something] next to [something],107,107,0.741223,True,True,0.000007,...,0.000043,0.000005,0.000376,0.000015,0.000021,0.000001,0.000010,0.000064,0.000001,0.000003


#### Main Analysis 
Comparison of:
- P(label=3|session=plug_1) -- should be high.
- P(label=3|session=plug_2) -- if its high it means plug as distractor is increasing probabilty of label=3.
- P(label=3|session not like 'plug%')

In [23]:
distractor = 'plug'
label = 3

prob_col = f'prob_{label}'

df['session_temp'] = np.where(df['session'] == f'{distractor}_1', "Normal",
                     np.where(df['session'] == f'{distractor}_2', f"distractor_{distractor}", 'Baseline'))
              
df.groupby(['session','session_temp']).agg(
    videos=('video_id', 'count'),
).reset_index()


,session,session_temp,videos
0,attaching_1,Baseline,9
1,books_2,Baseline,45
2,lady_2,Baseline,45
3,magnet_2,Baseline,45
4,piling_1,Baseline,9
5,plug_2,distractor_plug,45
6,plugging_1,Baseline,9
7,showing_photo_of_1,Baseline,9


In [24]:
pd.set_option('display.float_format',  '{:,.5f}'.format)

# Analysis special

### "special objects in common actions":

- plug_2 shows 15 videos of common actions being performed with object plug. We should see a spike in 'Plugging [something] into [something]' for these. 
- So plug_2 is actually where plug is a distractor.

- I have plug_2, lady_2, magnet_2, books_2 for "special objects in common actions" in recording session 2

In [32]:
def avg_prob_in_diff_cases(df: pd.DataFrame, distractor: str, label: str, label2id_dict: dict, id2label_dict:dict, cams: list):
    label_id = label2id_dict[label]
    prob_col = f'prob_{label_id}'

    df_temp = df[df['cam'].isin(cams)].copy()
    df_temp['session_temp'] = np.where(df_temp['session'] == f'{distractor}_1', "Normal",
                        np.where(df_temp['session'] == f'{distractor}_2', f"distractor_{distractor}", 'Baseline'))
    
    r = df_temp.groupby('session_temp').agg(
        videos=('video_id', 'count'),
        prob_mean=(prob_col, 'mean'),
        prob_std=(prob_col, 'std'),
        top1acc=('correct_top1', 'mean'),
        examples=('video_id', lambda x: ', '.join(map(str, x.unique()[:3]))),
    ).sort_values('session_temp').reset_index()

    print("=" * 25, distractor, "=" * 25)
    print(f"label {label_id}: {label}")
    print(f"cams used: {cams}")
    print(r)
    print("=" * 20, 'Videos with distractor', "=" * 20)

    dist_df = df_temp[df_temp['session_temp'] == f"distractor_{distractor}"] ### WHICH VIDEOS TO PRINT

    # ------------- chunk to copy videos ------------- #
    source_dir = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/"
    dest_dir = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/vjepa_evaluation_on_ssv2/analysis_videos_visualized/"

    # 2. Clear destination directory if it exists, then recreate it
    if os.path.exists(dest_dir):
        shutil.rmtree(dest_dir)
    os.makedirs(dest_dir, exist_ok=True)

    # 3. Get unique video IDs and copy matching files
    unique_video_ids = dist_df['video_id'].dropna().astype(str).unique()

    for filename in os.listdir(source_dir):
        if any(video_id in filename for video_id in unique_video_ids):
            shutil.copy(os.path.join(source_dir, filename), os.path.join(dest_dir, filename))
    # ------------------------------------------------ #

    prob_cols = [c for c in dist_df.columns if c.startswith('prob_')]
    meta_cols = [c for c in dist_df.columns if c not in prob_cols]

    for idx, row in dist_df.iterrows():
        top5_s = row[prob_cols].astype(float).nlargest(5)
        top5_cols = top5_s.index.tolist()
        
        # 1. Print row table as before (meta columns + top 5 prob_X headers)
        print(row[meta_cols + top5_cols].to_frame().T)
        
        # 2. Print mapping and probability cleanly in 5 separate lines
        print("\nTop 5 Probability Breakdown:")
        for col, val in top5_s.items():
            class_id = int(col.split('_')[1])
            class_name = id2label_dict.get(class_id, id2label_dict.get(str(class_id), 'Unknown'))
            print(f"  {col:<8} ->  {val:.5f}  |  {class_name}")

        print(f"\n  {prob_col:<8} ->  {row[prob_col]:.5f}  |  {label}")
            
        print("-" * 50)

In [33]:
avg_prob_in_diff_cases(df, distractor='plug', label='Plugging [something] into [something]', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= plug =========================
label 49: Plugging [something] into [something]
cams used: ['cam1']
      session_temp  videos  prob_mean  prob_std  top1acc                                     examples
0         Baseline      57    0.03573   0.14708  0.66667  260930-124820, 260930-124846, 260930-124905
1  distractor_plug      15    0.00080   0.00217  0.80000  260930-122254, 260930-122317, 260930-122340
==================== Videos with distractor ====================
        video_id   cam session                                 template label top1_pred top1_conf correct_top1 correct_top5     session_temp prob_107 prob_19 prob_114 prob_42 prob_109
1  260930-122254  cam1  plug_2  Putting [something] next to [something]   107       107   0.74122         True         True  distractor_plug  0.74122 0.07572  0.06682 0.05044  0.01915

Top 5 Probability Breakdown:
  prob_107 ->  0.74122  |  Putting [something] next to [something]
  prob_19  ->  0.07572  |  Holding [som

In [37]:
avg_prob_in_diff_cases(df, distractor='lady', label='Showing a photo of [something] to the camera', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= lady =========================
label 124: Showing a photo of [something] to the camera
cams used: ['cam1']
      session_temp  videos  prob_mean  prob_std  top1acc                                     examples
0         Baseline      57    0.04837   0.20676  0.70175  260930-122254, 260930-122317, 260930-122340
1  distractor_lady      15    0.00030   0.00041  0.66667  260930-124820, 260930-124846, 260930-124905
==================== Videos with distractor ====================
         video_id   cam session                                 template label top1_pred top1_conf correct_top1 correct_top5     session_temp prob_107 prob_114 prob_19 prob_73 prob_126
46  260930-124820  cam1  lady_2  Putting [something] next to [something]   107       107   0.98203         True         True  distractor_lady  0.98203  0.01091 0.00333 0.00126  0.00072

Top 5 Probability Breakdown:
  prob_107 ->  0.98203  |  Putting [something] next to [something]
  prob_114 ->  0.01091  |  Pu

In [38]:
avg_prob_in_diff_cases(df, distractor='magnet', label='Attaching [something] to [something]', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= magnet =========================
label 1: Attaching [something] to [something]
cams used: ['cam1']
        session_temp  videos  prob_mean  prob_std  top1acc                                     examples
0           Baseline      57    0.05704   0.18328  0.64912  260930-122254, 260930-122317, 260930-122340
1  distractor_magnet      15    0.02791   0.07730  0.86667  260930-141745, 260930-141807, 260930-141829
==================== Videos with distractor ====================
         video_id   cam   session                                 template label top1_pred top1_conf correct_top1 correct_top5       session_temp prob_107 prob_73 prob_19 prob_104 prob_114
91  260930-141745  cam1  magnet_2  Putting [something] next to [something]   107       107   0.97925         True         True  distractor_magnet  0.97925 0.00660 0.00243  0.00221  0.00189

Top 5 Probability Breakdown:
  prob_107 ->  0.97925  |  Putting [something] next to [something]
  prob_73  ->  0.00660 

In [36]:
avg_prob_in_diff_cases(df, distractor='books', label='Piling [something] up', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= books =========================
label 48: Piling [something] up
cams used: ['cam1']
       session_temp  videos  prob_mean  prob_std  top1acc                                     examples
0          Baseline      57    0.00317   0.01438  0.77193  260930-122254, 260930-122317, 260930-122340
1  distractor_books      15    0.00369   0.01007  0.40000  260930-143319, 260930-143358, 260930-143410
==================== Videos with distractor ====================
          video_id   cam  session                                 template label top1_pred top1_conf correct_top1 correct_top5      session_temp prob_114 prob_107 prob_19 prob_102 prob_103
136  260930-143319  cam1  books_2  Putting [something] next to [something]   107       114   0.49150        False         True  distractor_books  0.49150  0.35959 0.05690  0.02861  0.00581

Top 5 Probability Breakdown:
  prob_114 ->  0.49150  |  Putting [something similar to other things that are already on the table]
  prob_

### A closer look
##### at individual examples and try to spot the ones where the distractor seems to work.
##### ***Beware that sigmoid_logits are used as probability

In [126]:
df = result[['video_id',	'cam'	,'session'	,'template',	'label'	,'top1_pred','top1_conf','correct_top1']]
print(df.shape)
df.head(2)

(150, 8)


,video_id,cam,session,template,label,top1_pred,top1_conf,correct_top1
0,260922-134214,cam0,plug_1,Plugging [something] into [something],49,49,0.70513,True
1,260922-134214,cam1,plug_1,Plugging [something] into [something],49,49,0.77079,True


In [127]:
def sigmoid(x):
    return 1/(1+np.exp(-x))
    
d_sigmoid_logits = sigmoid(d['logits'])
d_sigmoid_logits.shape

(150, 174)

In [128]:
# Define class probability column names
class_cols = [f'prob_{i}' for i in range(174)]

# Convert probabilities array into a DataFrame using matching index
# probs_df = pd.DataFrame(d['probs'], columns=class_cols, index=df.index)
probs_df = pd.DataFrame(d_sigmoid_logits, columns=class_cols, index=df.index)

# Combine metadata with class probabilities
df = pd.concat([df, probs_df], axis=1)

pd.set_option('display.float_format',  '{:,.6f}'.format)

df.head(2)

,video_id,cam,session,template,label,top1_pred,top1_conf,correct_top1,prob_0,prob_1,...,prob_164,prob_165,prob_166,prob_167,prob_168,prob_169,prob_170,prob_171,prob_172,prob_173
0,260922-134214,cam0,plug_1,Plugging [something] into [something],49,49,0.705132,True,0.106691,0.999704,...,0.460774,0.125065,0.370225,0.171061,0.214691,0.597843,0.924142,0.004070,0.000536,0.103749
1,260922-134214,cam1,plug_1,Plugging [something] into [something],49,49,0.770786,True,0.205615,0.999722,...,0.496765,0.189521,0.320821,0.169956,0.240775,0.337204,0.933431,0.005555,0.000688,0.174410


In [123]:
def avg_prob_in_diff_cases(df: pd.DataFrame, distractor: str, label: str, label2id_dict: dict, id2label_dict:dict, cams: list):
    label_id = label2id_dict[label]
    prob_col = f'prob_{label_id}'

    df_temp = df[df['cam'].isin(cams)].copy()
    df_temp['session_temp'] = np.where(df_temp['session'] == f'{distractor}_1', "Normal",
                        np.where(df_temp['session'] == f'{distractor}_2', f"distractor_{distractor}", 'Baseline'))
    
    r = df_temp.groupby('session_temp').agg(
        videos=('video_id', 'count'),
        prob_mean=(prob_col, 'mean'),
        prob_std=(prob_col, 'std'),
        top1acc=('correct_top1', 'mean'),
        examples=('video_id', lambda x: ', '.join(map(str, x.unique()[:3]))),
    ).sort_values('session_temp').reset_index()

    print("=" * 25, distractor, "=" * 25)
    print(f"label {label_id}: {label}")
    print(f"cams used: {cams}")
    print(r)
    
    print("=" * 20, 'Videos with distractor', "=" * 20)
    dist_df = df_temp[df_temp['session_temp'] == f"distractor_{distractor}"]
    prob_cols = [c for c in dist_df.columns if c.startswith('prob_')]
    meta_cols = [c for c in dist_df.columns if c not in prob_cols]

    for idx, row in dist_df.iterrows():
        top5_cols = row[prob_cols].astype(float).nlargest(5).index.tolist()
        print(row[meta_cols + top5_cols].to_frame().T)
        print('-'*10)

avg_prob_in_diff_cases(df, distractor='plug', label='Plugging [something] into [something]', label2id_dict=label2id_dict, cams=['cam1'])

========================= plug =========================
label 49: Plugging [something] into [something]
cams used: ['cam1']
      session_temp  videos  prob_mean  prob_std  top1acc                                     examples
0         Baseline      40    0.64286   0.22287  0.55000  260922-143234, 260922-143255, 260922-143323
1           Normal       5    0.99996   0.00002  1.00000  260922-134214, 260922-134301, 260922-134333
2  distractor_plug       5    0.40757   0.27606  0.80000  260922-141236, 260922-141249, 260922-141308
==================== Videos with distractor ====================
         video_id   cam session                                 template label top1_pred top1_conf correct_top1     session_temp prob_114 prob_107 prob_42  prob_6 prob_105
16  260922-141236  cam1  plug_2  Putting [something] next to [something]   107       114   0.70823        False  distractor_plug  0.99975  0.99887 0.99265 0.98999  0.98795
----------
         video_id   cam session                

In [124]:
def avg_prob_in_diff_cases(df: pd.DataFrame, distractor: str, label: str, label2id_dict: dict, id2label_dict:dict, cams: list):
    label_id = label2id_dict[label]
    prob_col = f'prob_{label_id}'

    df_temp = df[df['cam'].isin(cams)].copy()
    df_temp['session_temp'] = np.where(df_temp['session'] == f'{distractor}_1', "Normal",
                        np.where(df_temp['session'] == f'{distractor}_2', f"distractor_{distractor}", 'Baseline'))
    
    r = df_temp.groupby('session_temp').agg(
        videos=('video_id', 'count'),
        prob_mean=(prob_col, 'mean'),
        prob_std=(prob_col, 'std'),
        top1acc=('correct_top1', 'mean'),
        examples=('video_id', lambda x: ', '.join(map(str, x.unique()[:3]))),
    ).sort_values('session_temp').reset_index()

    print("-" * 25, distractor, "-" * 25)
    print(f"label {label_id}: {label}")
    print(f"cams used: {cams}")
    print(r)
    
    print("-" * 20, 'Videos with distractor', "-" * 20)
    dist_df = df_temp[df_temp['session_temp'] == f"distractor_{distractor}"]
    prob_cols = [c for c in dist_df.columns if c.startswith('prob_')]
    meta_cols = [c for c in dist_df.columns if c not in prob_cols]

    for idx, row in dist_df.iterrows():
        top5_cols = row[prob_cols].astype(float).nlargest(5).index.tolist()
        
        # Rename top 5 prob columns to include full class label name
        rename_map = {}
        for col in top5_cols:
            class_id = int(col.split('_')[1])
            class_name = id2label_dict.get(class_id, id2label_dict.get(str(class_id), col))
            rename_map[col] = f"{col}: {class_name}"

        row_df = row[meta_cols + top5_cols].to_frame().T.rename(columns=rename_map)
        print(row_df)
        print("-" * 50)

avg_prob_in_diff_cases(df, distractor='plug', label='Plugging [something] into [something]', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

------------------------- plug -------------------------
label 49: Plugging [something] into [something]
cams used: ['cam1']
      session_temp  videos  prob_mean  prob_std  top1acc                                     examples
0         Baseline      40    0.64286   0.22287  0.55000  260922-143234, 260922-143255, 260922-143323
1           Normal       5    0.99996   0.00002  1.00000  260922-134214, 260922-134301, 260922-134333
2  distractor_plug       5    0.40757   0.27606  0.80000  260922-141236, 260922-141249, 260922-141308
-------------------- Videos with distractor --------------------
         video_id   cam session                                 template label top1_pred top1_conf correct_top1     session_temp prob_114: Putting [something similar to other things that are already on the table] prob_107: Putting [something] next to [something] prob_42: Moving [something] closer to [something] prob_6: Covering [something] with [something] prob_105: Putting [something] in front of [